### Demo Self Attention

#### Imports

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple
                           

#### Configurations

In [6]:
@dataclass
class Config:
    line_divider: str = '-' * 50
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    random_seed: int = 42  # For reproducibility

config = Config()                                                                                              

#### Self-Attention Class

In [7]:
torch.manual_seed(config.random_seed)

class SelfAttention(nn.Module):
    def __init__(self, d_model=2,  
                 row_dim=0, 
                 col_dim=1):
        super(SelfAttention, self).__init__()
        self.d_model = d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)  
        self.W_v = nn.Linear(d_model, d_model)
        self.row_dim = row_dim
        self.col_dim = col_dim

    def forward(self, token_encodings):
        batch_size = 1
        seq_len, d_model = token_encodings.size()
        Q = self.W_q(token_encodings)  
        K = self.W_k(token_encodings)  
        V = self.W_v(token_encodings)  
        scores = torch.matmul(Q, K.transpose(dim0=self.row_dim, dim1= self.col_dim))
        scores_scaled = scores / (self.d_model ** 0.5)
        attn_weights = F.softmax(scores_scaled, dim=self.col_dim)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

#### Calculate Self-Attention

In [5]:
embed_dim = 4
text = "The cat sat on the mat, it was black."
tokens = text.split()
token_encodings = torch.randn(len(tokens), embed_dim)
self_attention = SelfAttention(d_model=embed_dim)
output, attn_weights = self_attention(token_encodings)
print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)
print(f"{config.line_divider}\n")
print("Output:", output)
print(f"{config.line_divider}\n")
print("Attention weights:", attn_weights)
print(f"{config.line_divider}\n")


Output shape: torch.Size([9, 4])
Attention weights shape: torch.Size([9, 9])
--------------------------------------------------

Output: tensor([[ 0.3099, -0.2765, -0.1676, -0.1584],
        [ 0.3089, -0.2725, -0.1591, -0.1491],
        [ 0.2552, -0.2998, -0.1693, -0.1809],
        [ 0.2713, -0.3425, -0.0983, -0.1981],
        [ 0.3034, -0.3249, -0.1169, -0.1922],
        [ 0.2908, -0.3052, -0.1412, -0.1774],
        [ 0.2987, -0.2337, -0.2279, -0.1380],
        [ 0.3088, -0.2010, -0.2698, -0.1194],
        [ 0.3108, -0.3141, -0.1249, -0.1845]], grad_fn=<MmBackward0>)
--------------------------------------------------

Attention weights: tensor([[0.1058, 0.1108, 0.1572, 0.1092, 0.0782, 0.1080, 0.1169, 0.1375, 0.0763],
        [0.1026, 0.1070, 0.1523, 0.0978, 0.0778, 0.1028, 0.1347, 0.1468, 0.0782],
        [0.1147, 0.1033, 0.0870, 0.0977, 0.1341, 0.1075, 0.1103, 0.1085, 0.1368],
        [0.1138, 0.0746, 0.1284, 0.0975, 0.1365, 0.1118, 0.0910, 0.1230, 0.1235],
        [0.1087, 0.0794, 0